# 03 — Capacity Planning & Headroom Analysis

## Portfolio objective

Translate the forecast capacity footprint into an explicit planning requirement and identify when the current baseline becomes insufficient.

### Planning assumptions

These are **synthetic portfolio assumptions**, not Alibaba or Intel operational data:

- Target utilization = 70%
- Usable capacity factor = 90%
- Safety buffer = 10%
- Standard expansion unit = 8 GPUs/server

The planning factor is applied once in this stage.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from cpoe.config import PROCESSED, FIGURES
from cpoe.capacity.planning import calculate_capacity_requirements

forecast = pd.read_csv(
    PROCESSED / "workload_forecast.csv",
    index_col=0,
    parse_dates=True,
)
history = pd.read_csv(
    PROCESSED / "hourly_capacity_footprint.csv",
    index_col=0,
    parse_dates=True,
)

capacity, summary, planning_factor = calculate_capacity_requirements(
    forecast, history
)

print("Planning factor:", planning_factor)
display(summary)


In [ ]:
display(capacity.head())

capacity.to_csv(PROCESSED / "capacity_requirements.csv")
summary.to_csv(PROCESSED / "capacity_summary.csv")


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(capacity.index, capacity["forecast_demand"], label="Forecast capacity footprint")
plt.plot(capacity.index, capacity["required_capacity"], label="Required planning capacity")
plt.axhline(
    capacity["baseline_capacity"].iloc[0],
    linestyle="--",
    label="Latest observed baseline",
)
plt.title("Capacity Footprint vs Planning Requirement")
plt.xlabel("Trace-relative time")
plt.ylabel("GPU-equivalent capacity")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / "capacity_requirement.png", dpi=150)
plt.show()


## Decision question

> Given the forecast capacity footprint and explicit planning assumptions, how much GPU-equivalent capacity should be planned for, and when does the current baseline become insufficient?

Notebook 04 converts this requirement into an economically constrained expansion decision.
